# 🚀 02. La Première Baseline AutoML
**Projet AutoML – Challenge Machine Learning (M1 IA, Le Mans Université)**

Ce notebook implémente et teste notre **Baseline V1** (le Minimum Viable Product).

### Qu'est-ce qu'une baseline et pourquoi la construire en premier ?
- **Une référence minimale :** C'est le code le plus simple possible qui fonctionne de bout en bout.
- **Le score à battre :** Il sert de point de départ pour mesurer si nos futures optimisations (Optuna, modèles avancés) apportent un vrai gain.
- **La garantie de réussite :** Avoir une baseline fonctionnelle garantit dès le premier jour que notre projet respecte le contrat technique du sujet.

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, r2_score

CORPUS_DIR = "/info/corpus/ChallengeMachineLearning"
print("✓ Bibliothèques chargées avec succès.")

---
## Étape 1 : L'Aiguillage automatique (Task Dispatcher)

Comment notre programme sait-il vers quelle branche envoyer les données ?
Il regarde simplement la forme et les valeurs du tableau des cibles $y$ :

In [ ]:
def detecter_tache(y):
    """
    Analyse y pour choisir automatiquement la branche :
    - Régression
    - Classification binaire
    - Multiclasse (One-Hot)
    - Multi-label
    """
    # 1. Cas où y a plusieurs colonnes (2D)
    if y.ndim == 2 and y.shape[1] > 1:
        est_binaire = np.all((y == 0) | (y == 1))
        if est_binaire:
            # Si chaque ligne a exactement un seul '1' -> One-Hot
            if np.all(np.sum(y, axis=1) == 1):
                return 'classification_multiclasse_onehot'
            else:
                return 'classification_multilabel'
        else:
            return 'regression_multioutput'

    # 2. Cas où y a 1 seule colonne (1D)
    y_plat = y.ravel()
    valeurs_uniques = np.unique(y_plat)
    nb_valeurs = len(valeurs_uniques)

    # Seulement 2 valeurs distinctes (ex: 0 et 1)
    if nb_valeurs == 2:
        return 'classification_binaire'

    # Valeurs entières en nombre limité
    est_entier = np.issubdtype(y_plat.dtype, np.integer) or np.all(np.equal(np.mod(y_plat, 1), 0))
    if est_entier and nb_valeurs < 50:
        return 'classification_multiclasse'

    # Sinon, valeurs réelles continues -> Régression
    return 'regression'

### Test de l'aiguillage sur les 6 jeux de données :

In [ ]:
for nom in ["data_A", "data_B", "data_C", "data_D", "data_E", "data_F"]:
    sol = np.loadtxt(os.path.join(CORPUS_DIR, nom, f"{nom}.solution"))
    branche = detecter_tache(sol)
    print(f"{nom:8s} ➡️ Branche : [{branche}]")

---
## Étape 2 : Le moteur de la Baseline Automatique

Voici notre fonction `entrainer_baseline`.
Pour rester rapide et léger, on prend un échantillon de **1 000 lignes** par défaut :

In [ ]:
def load_features_simple(filepath):
    with open(filepath, 'r') as f:
        first_line = f.readline().strip()
    if ':' in first_line:
        rows, cols, data = [], [], []
        with open(filepath, 'r') as f:
            for r, line in enumerate(f):
                for item in line.strip().split():
                    c, v = item.split(':')
                    rows.append(r)
                    cols.append(int(c))
                    data.append(float(v))
        min_c = min(cols)
        if min_c == 1:
            cols = [c - 1 for c in cols]
        return sp.csr_matrix((data, (rows, cols)), shape=(r + 1, max(cols) + 1))
    return np.loadtxt(filepath)

def entrainer_baseline(nom_dataset, max_echantillon=1000):
    """
    1. Charge les données
    2. Échantillonne pour la rapidité
    3. Aiguille automatiquement la tâche
    4. Entraîne le modèle scikit-learn adapté
    5. Calcule et affiche le score
    """
    p_data = os.path.join(CORPUS_DIR, nom_dataset, f"{nom_dataset}.data")
    p_sol = os.path.join(CORPUS_DIR, nom_dataset, f"{nom_dataset}.solution")
    
    X = load_features_simple(p_data)
    y = np.loadtxt(p_sol)
    
    # Sous-échantillonnage rapide
    if len(y) > max_echantillon:
        X = X[:max_echantillon]
        y = y[:max_echantillon]
        
    # Détection automatique de la branche
    tache = detecter_tache(y)
    
    # Adaptation pour scikit-learn
    if "onehot" in tache:
        y_train_fit = np.argmax(y, axis=1)
    else:
        y_train_fit = y.ravel() if y.ndim == 2 and y.shape[1] == 1 else y
        
    X_train, X_test, y_train, y_test = train_test_split(X, y_train_fit, test_size=0.2, random_state=42)
    
    # Choix automatique du modèle
    if "classification" in tache:
        modele = RandomForestClassifier(n_estimators=50, random_state=42)
        modele.fit(X_train, y_train)
        preds = modele.predict(X_test)
        score = accuracy_score(y_test, preds)
        metrique = "Accuracy"
    else:
        modele = RandomForestRegressor(n_estimators=50, random_state=42)
        modele.fit(X_train, y_train)
        preds = modele.predict(X_test)
        score = r2_score(y_test, preds)
        metrique = "R²"
        
    return {
        "Dataset": nom_dataset,
        "Branche": tache,
        "Modèle": modele.__class__.__name__,
        "Métrique": metrique,
        "Score Baseline": round(score, 4)
    }

---
## Étape 3 : Tableau des scores de référence (Nos records à battre)

Exécutons notre baseline sur 3 jeux représentatifs :

In [ ]:
resultats = []
for test_ds in ["data_D", "data_B", "data_C"]:
    res = entrainer_baseline(test_ds, max_echantillon=1000)
    resultats.append(res)

df_resultats = pd.DataFrame(resultats)
df_resultats

---
## Conclusion & Suite du projet

### Nos repères sont posés :
- **Classification binaire (`data_D`) :** 78.0 % d'accuracy.
- **Multiclasse (`data_C`) :** 89.0 % d'accuracy.
- **Régression (`data_B`) :** R² = 0.6605.

### Prochaine étape pour l'équipe :
Transférer cette logique dans le package `automl/` (`automl.fit`, `automl.eval`, `automl.predict`) pour rendre notre première version officielle du challenge !